# CSP Screener - Interactive Dashboard

Cash-Secured Put opportunity scanner with Yahoo Finance data.

In [ ]:
# Core imports
import time
import yfinance as yf
from datetime import date
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from math import log, sqrt
from scipy.stats import norm

# Default watchlist
DEFAULT_WATCHLIST = [
    "AAPL", "AMZN", "AVGO", "BRK-B", "GOOG", "META", "MSFT", "NFLX", 
    "NVDA", "ORCL", "PEP", "PLTR", "TSLA", "TSM", "U", "UNH", "V"
]

print("✓ Libraries loaded")

✓ Libraries loaded


## Helper Functions

In [2]:
def black_scholes_put_delta(s: float, k: float, t: float, r: float = 0.05, sigma: float = 0.30) -> float:
    """Black-Scholes put delta."""
    if t <= 0 or sigma <= 0:
        return -1.0 if s < k else 0.0
    d1 = (log(s / k) + (r + 0.5 * sigma**2) * t) / (sigma * sqrt(t))
    return norm.cdf(d1) - 1


def calc_rating(iv: float, delta: float, dte: int, ann_roc: float, strong_fundamentals: bool) -> tuple[int, str]:
    """Compute composite rating score and label."""
    pts = 0
    
    # IV scoring
    if iv >= 0.60:
        pts += 2
    elif iv >= 0.30:
        pts += 1
    else:
        pts -= 1
    
    # Delta scoring
    if 0.20 <= delta <= 0.30:
        pts += 2
    elif delta < 0.15 or delta > 0.35:
        pts -= 1
    
    # DTE scoring
    if 30 <= dte <= 45:
        pts += 2
    
    # ROC scoring
    if ann_roc >= 20:
        pts += 2
    elif ann_roc >= 12:
        pts += 1
    
    # Fundamentals bonus
    if strong_fundamentals:
        pts += 1
    
    score = max(1, min(5, pts))
    label = {5: "STRONG", 4: "GOOD", 3: "OK", 2: "FAIR", 1: "POOR"}.get(score, "OK")
    return score, label


def is_strong_fundamentals(pe_ratio, profit_margin, beta) -> bool:
    """Check if fundamentals meet quality criteria."""
    if pe_ratio is None or profit_margin is None or beta is None:
        return False
    return pe_ratio > 0 and profit_margin > 10 and beta < 1.5


print("✓ Helper functions defined")

✓ Helper functions defined


## Scanner Function

In [3]:
def scan_ticker(
    symbol: str,
    min_dte: int,
    max_dte: int,
    min_otm_pct: float,
    min_iv: float,
    min_delta: float,
    max_delta: float,
    min_ann_roc: float,
    max_beta: float,
    chain_delay: float = 1.5,
) -> tuple[list[dict], str | None]:
    """Scan a single ticker for CSP opportunities.
    
    Returns (opportunities, error_message).
    """
    opportunities = []
    
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.info
        
        if not info:
            return [], "No info returned"
        
        price = info.get("currentPrice") or info.get("regularMarketPrice")
        if not price:
            return [], "No price available"
        
        # Fundamentals
        pe_ratio = info.get("trailingPE")
        beta = info.get("beta")
        profit_margin = info.get("profitMargins")
        if profit_margin is not None:
            profit_margin = profit_margin * 100
        revenue_growth = info.get("revenueGrowth")
        if revenue_growth is not None:
            revenue_growth = revenue_growth * 100
        
        strong = is_strong_fundamentals(pe_ratio, profit_margin, beta)
        
        # Filter by beta
        if beta is not None and beta > max_beta:
            return [], f"Beta {beta:.2f} > max {max_beta}"
        
        # Get expirations
        expirations = ticker.options
        if not expirations:
            return [], "No options data"
        
        today = date.today()
        valid_expirations = []
        
        for exp_str in expirations:
            try:
                exp_date = date.fromisoformat(exp_str)
                dte = (exp_date - today).days
                if min_dte <= dte <= max_dte:
                    valid_expirations.append((exp_str, dte))
            except ValueError:
                continue
        
        # Limit to first 3 expirations to reduce API calls
        valid_expirations = valid_expirations[:3]
        
        if not valid_expirations:
            return [], f"No expirations in DTE range {min_dte}-{max_dte}"
        
        # Scan each expiration
        for i, (exp_str, dte) in enumerate(valid_expirations):
            if i > 0:
                time.sleep(chain_delay)
            
            try:
                chain = ticker.option_chain(exp_str)
            except Exception:
                continue
            
            if chain.puts is None or chain.puts.empty:
                continue
            
            for _, put in chain.puts.iterrows():
                strike = put.get("strike")
                bid = put.get("bid")
                iv = put.get("impliedVolatility")
                ask = put.get("ask")
                
                if strike is None or bid is None or bid <= 0 or iv is None or iv <= 0:
                    continue
                
                # Calculate metrics
                mid = (bid + (ask or bid)) / 2
                otm_pct = (price - strike) / price * 100
                ann_roc = (bid / strike) / dte * 365 * 100
                capital = strike * 100
                
                # Calculate delta
                time_to_expiry = dte / 365
                delta_raw = black_scholes_put_delta(price, strike, time_to_expiry, 0.05, iv)
                delta = abs(delta_raw)
                
                # Apply filters
                if otm_pct < min_otm_pct:
                    continue
                if iv < min_iv:
                    continue
                if delta < min_delta or delta > max_delta:
                    continue
                if ann_roc < min_ann_roc:
                    continue
                
                # Calculate rating
                rating, label = calc_rating(iv, delta, dte, ann_roc, strong)
                
                opportunities.append({
                    "symbol": symbol,
                    "price": round(price, 2),
                    "strike": strike,
                    "expiry": exp_str,
                    "dte": dte,
                    "bid": round(bid, 2),
                    "mid": round(mid, 2),
                    "iv": round(iv, 4),
                    "delta": round(delta, 4),
                    "otm_pct": round(otm_pct, 2),
                    "ann_roc": round(ann_roc, 2),
                    "capital": capital,
                    "pe_ratio": round(pe_ratio, 2) if pe_ratio else None,
                    "beta": round(beta, 2) if beta else None,
                    "profit_margin": round(profit_margin, 2) if profit_margin else None,
                    "revenue_growth": round(revenue_growth, 2) if revenue_growth else None,
                    "strong_fundamentals": strong,
                    "rating": rating,
                    "rating_label": label,
                })
        
        return opportunities, None
        
    except Exception as e:
        err_msg = str(e).lower()
        if "rate" in err_msg or "429" in err_msg or "too many" in err_msg:
            return [], "RATE LIMITED"
        return [], str(e)


print("✓ Scanner function defined")

✓ Scanner function defined


## Step 1: Scan Market Data

Fetch put options data from Yahoo Finance. Run once, then use Step 2 to filter.

In [ ]:
# ============================================================================
# CELL 1: SCAN - Fetch data from Yahoo Finance
# Run this once to populate scan_cache with raw data
# ============================================================================

# Style
style = {'description_width': '120px'}
layout = widgets.Layout(width='300px')

# Scan configuration widgets
min_dte_scan = widgets.IntSlider(value=21, min=7, max=60, step=1, description='Min DTE:', style=style, layout=layout)
max_dte_scan = widgets.IntSlider(value=45, min=14, max=90, step=1, description='Max DTE:', style=style, layout=layout)
delay_scan = widgets.FloatSlider(value=5.0, min=1, max=15, step=0.5, description='Delay (s):', style=style, layout=layout)

watchlist_text = widgets.Textarea(
    value=", ".join(DEFAULT_WATCHLIST[:10]),
    description='Watchlist:',
    placeholder='ADBE, AMZN, NVDA, ...',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='500px', height='60px')
)

scan_btn = widgets.Button(description='Scan', button_style='primary', layout=widgets.Layout(width='120px'))
clear_btn = widgets.Button(description='Clear Cache', button_style='warning', layout=widgets.Layout(width='100px'))

scan_status = widgets.Output()

# Global cache for scan results
scan_cache = {'data': [], 'failed': [], 'timestamp': None}

def get_watchlist():
    text = watchlist_text.value.strip()
    if not text:
        return []
    return [s.strip().upper() for s in text.replace('\n', ',').split(',') if s.strip()]

def run_scan(b):
    """Fetch ALL opportunities - no filtering, just raw data collection."""
    global scan_cache
    
    symbols = get_watchlist()
    if not symbols:
        with scan_status:
            clear_output()
            print("No symbols in watchlist")
        return
    
    all_opps = []
    failed = []
    
    with scan_status:
        clear_output()
        print(f"Scanning {len(symbols)} tickers (DTE {min_dte_scan.value}-{max_dte_scan.value})...\n")
        
        for i, symbol in enumerate(symbols):
            print(f"[{i+1}/{len(symbols)}] {symbol}...", end=" ")
            
            if i > 0:
                time.sleep(delay_scan.value)
            
            # Fetch with minimal filters - get everything, filter later
            opps, error = scan_ticker(
                symbol, 
                min_dte=min_dte_scan.value, 
                max_dte=max_dte_scan.value,
                min_otm_pct=0,
                min_iv=0,
                min_delta=0,
                max_delta=1.0,
                min_ann_roc=0,
                max_beta=100.0,
                chain_delay=1.5,
            )
            
            if error:
                if error == "RATE LIMITED":
                    print("RATE LIMITED")
                else:
                    print(f"SKIP: {error}")
                failed.append(symbol)
            else:
                print(f"{len(opps)} puts")
                all_opps.extend(opps)
        
        scan_cache = {
            'data': all_opps,
            'failed': failed,
            'timestamp': date.today().isoformat()
        }
        
        print(f"\n{'='*50}")
        print(f"Cached {len(all_opps)} total opportunities")
        print(f"Failed: {len(failed)} tickers")
        print(f"Now use the filter cell below to analyze.")

def clear_cache(b):
    global scan_cache
    scan_cache = {'data': [], 'failed': [], 'timestamp': None}
    with scan_status:
        clear_output()
        print("Cache cleared.")

scan_btn.on_click(run_scan)
clear_btn.on_click(clear_cache)

# Display scan UI
display(widgets.VBox([
    widgets.HTML('<h3>Scan Market Data</h3>'),
    widgets.HBox([min_dte_scan, max_dte_scan, delay_scan]),
    widgets.HBox([watchlist_text]),
    widgets.HBox([scan_btn, clear_btn]),
    scan_status,
]))

## Step 2: Filter & Analyze Results

Adjust sliders to filter cached data instantly (no API calls). Re-run this cell anytime to reset filters.

In [5]:
# ============================================================================
# CELL 2: FILTER & ANALYZE - Works on cached data
# Run this cell after scanning to get interactive filtering UI
# Adjust sliders to filter cached results instantly (no API calls)
# ============================================================================

# Filter widgets
style = {'description_width': '120px'}
layout = widgets.Layout(width='280px')

min_iv_filter = widgets.FloatSlider(value=30, min=0, max=100, step=5, description='Min IV (%):', style=style, layout=layout)
min_dte_filter = widgets.IntSlider(value=21, min=7, max=90, step=1, description='Min DTE:', style=style, layout=layout)
max_dte_filter = widgets.IntSlider(value=45, min=14, max=90, step=1, description='Max DTE:', style=style, layout=layout)
min_otm_filter = widgets.FloatSlider(value=5.0, min=0, max=25, step=0.5, description='Min OTM (%):', style=style, layout=layout)
min_delta_filter = widgets.FloatSlider(value=0.15, min=0.05, max=0.50, step=0.01, description='Min Delta:', style=style, layout=layout)
max_delta_filter = widgets.FloatSlider(value=0.35, min=0.10, max=0.50, step=0.01, description='Max Delta:', style=style, layout=layout)
min_roc_filter = widgets.FloatSlider(value=12.0, min=0, max=40, step=1, description='Min Ann ROC (%):', style=style, layout=layout)
max_beta_filter = widgets.FloatSlider(value=2.5, min=0.5, max=5.0, step=0.1, description='Max Beta:', style=style, layout=layout)
min_rating_filter = widgets.IntSlider(value=3, min=1, max=5, step=1, description='Min Rating:', style=style, layout=layout)

filter_output = widgets.Output()

def apply_filters():
    """Apply current filter settings to cached data."""
    if not scan_cache['data']:
        return pd.DataFrame()
    
    df = pd.DataFrame(scan_cache['data'])
    
    filtered = df[
        (df['iv'] >= min_iv_filter.value / 100) &
        (df['dte'] >= min_dte_filter.value) &
        (df['dte'] <= max_dte_filter.value) &
        (df['otm_pct'] >= min_otm_filter.value) &
        (df['delta'] >= min_delta_filter.value) &
        (df['delta'] <= max_delta_filter.value) &
        (df['ann_roc'] >= min_roc_filter.value) &
        (df['rating'] >= min_rating_filter.value)
    ].copy()
    
    # Beta filter (handle None)
    if 'beta' in df.columns:
        filtered = filtered[
            filtered['beta'].isna() | (filtered['beta'] <= max_beta_filter.value)
        ]
    
    return filtered.sort_values('ann_roc', ascending=False)

def update_display(change=None):
    """Update display when filters change."""
    with filter_output:
        clear_output()
        
        if not scan_cache['data']:
            print("No data in cache. Run the scan cell above first.")
            return
        
        df = apply_filters()
        
        if df.empty:
            print("No opportunities match current filters.")
            print(f"\nCache has {len(scan_cache['data'])} total opportunities.")
            print("Relax filters or run a new scan.")
            return
        
        # Summary
        print("SUMMARY")
        print(f"   Matching: {len(df)} of {len(scan_cache['data'])} cached")
        print(f"   Avg IV: {df['iv'].mean()*100:.1f}%")
        print(f"   Avg Ann ROC: {df['ann_roc'].mean():.1f}%")
        print(f"   Avg DTE: {df['dte'].mean():.0f}")
        print(f"   Total Capital: ${df['capital'].sum():,.0f}")
        if scan_cache['failed']:
            print(f"   Failed tickers: {', '.join(scan_cache['failed'])}")
        print()
        
        # Rating color styling
        def rating_style(val):
            colors = {
                'STRONG': 'background-color: #d4edda; color: #155724',
                'GOOD': 'background-color: #e2efda; color: #28a745',
                'OK': 'background-color: #fff3cd; color: #856404',
                'FAIR': 'background-color: #ffe5d0; color: #fd7e14',
                'POOR': 'background-color: #f8d7da; color: #721c24'
            }
            return colors.get(val, '')
        
        # Display table
        cols = ['symbol', 'strike', 'expiry', 'dte', 'bid', 'price', 'iv', 'delta', 'otm_pct', 'ann_roc', 'rating', 'rating_label']
        display_df = df[cols].copy()
        display_df['iv'] = (display_df['iv'] * 100).round(1)
        display_df.columns = ['Symbol', 'Strike', 'Expiry', 'DTE', 'Bid', 'Price', 'IV%', 'Delta', 'OTM%', 'ROC%', 'Rating#', 'Rating']
        
        styled = display_df.style.map(rating_style, subset=['Rating'])
        display(styled)
        
        # Top 5 detailed
        print("\nTOP 5 DETAILS")
        print("=" * 70)
        for _, row in df.head(5).iterrows():
            print(f"\n{row['symbol']} ${row['strike']} PUT | Exp: {row['expiry']} ({row['dte']} DTE)")
            print(f"   Stock: ${row['price']} | Strike OTM: {row['otm_pct']}% | IV: {row['iv']*100:.1f}%")
            print(f"   Bid: ${row['bid']} | Ann ROC: {row['ann_roc']}% | Capital: ${row['capital']:,.0f}")
            print(f"   Rating: {row['rating_label']} ({row['rating']}/5) | Delta: {row['delta']:.3f}")
            if row['beta']:
                print(f"   Beta: {row['beta']} | PE: {row['pe_ratio']} | Margin: {row['profit_margin']}%")

# Wire filter observers
for slider in [min_iv_filter, min_dte_filter, max_dte_filter, min_otm_filter,
               min_delta_filter, max_delta_filter, min_roc_filter, max_beta_filter,
               min_rating_filter]:
    slider.observe(update_display, names='value')

# Display UI
display(widgets.VBox([
    widgets.HTML('<h3>Filter & Analyze Results</h3>'),
    widgets.HTML('<p>Adjust sliders to filter cached data instantly (no API calls)</p>'),
    widgets.HBox([min_iv_filter, min_otm_filter]),
    widgets.HBox([min_dte_filter, max_dte_filter]),
    widgets.HBox([min_delta_filter, max_delta_filter]),
    widgets.HBox([min_roc_filter, max_beta_filter]),
    widgets.HBox([min_rating_filter]),
    filter_output,
]))

# Initial display
update_display()

In [ ]:
# ============================================================================
# CELL 3: Quick Analysis Tools
# Helper functions for deeper analysis on cached data
# ============================================================================

def show_by_symbol(symbol: str):
    """Show all cached opportunities for a specific symbol."""
    if not scan_cache['data']:
        print("Run scan first.")
        return
    
    df = pd.DataFrame(scan_cache['data'])
    subset = df[df['symbol'] == symbol.upper()].sort_values('ann_roc', ascending=False)
    
    if subset.empty:
        print(f"No opportunities for {symbol.upper()} in cache.")
    else:
        print(f"{symbol.upper()}: {len(subset)} opportunities\n")
        cols = ['strike', 'expiry', 'dte', 'bid', 'price', 'iv', 'delta', 'otm_pct', 'ann_roc', 'rating_label']
        display_df = subset[cols].copy()
        display_df['iv'] = (display_df['iv'] * 100).round(1)
        display_df.columns = ['Strike', 'Expiry', 'DTE', 'Bid', 'Price', 'IV%', 'Delta', 'OTM%', 'ROC%', 'Rating']
        display(display_df.style.map(lambda v: 'background-color: #e2efda' if v in ['STRONG', 'GOOD'] else '', subset=['Rating']))

def show_top_n(n: int = 10, by: str = 'ann_roc'):
    """Show top N opportunities sorted by specified metric."""
    if not scan_cache['data']:
        print("Run scan first.")
        return
    
    df = pd.DataFrame(scan_cache['data'])
    if by not in df.columns:
        print(f"Invalid sort field: {by}. Use: ann_roc, iv, otm_pct, dte, rating")
        return
    
    top = df.sort_values(by, ascending=False).head(n)
    print(f"Top {n} by {by}:\n")
    cols = ['symbol', 'strike', 'expiry', 'dte', 'bid', 'iv', 'otm_pct', 'ann_roc', 'rating_label']
    display_df = top[cols].copy()
    display_df['iv'] = (display_df['iv'] * 100).round(1)
    display_df.columns = ['Symbol', 'Strike', 'Expiry', 'DTE', 'Bid', 'IV%', 'OTM%', 'ROC%', 'Rating']
    display(display_df)

def group_by_symbol():
    """Summary grouped by symbol."""
    if not scan_cache['data']:
        print("Run scan first.")
        return
    
    df = pd.DataFrame(scan_cache['data'])
    summary = df.groupby('symbol').agg({
        'ann_roc': ['count', 'mean', 'max'],
        'iv': 'mean',
        'otm_pct': 'mean',
        'dte': 'mean',
        'rating': 'mean'
    }).round(2)
    summary.columns = ['Opps', 'Avg ROC%', 'Max ROC%', 'Avg IV%', 'Avg OTM%', 'Avg DTE', 'Avg Rating']
    summary = summary.sort_values('Max ROC%', ascending=False)
    display(summary)

def export_cache(filename: str = "screener_cache.csv"):
    """Export full cached results to CSV."""
    if not scan_cache['data']:
        print("Run scan first.")
        return
    
    df = pd.DataFrame(scan_cache['data'])
    df.to_csv(filename, index=False)
    print(f"Exported {len(df)} rows to {filename}")

def get_cache_df():
    """Return cached data as DataFrame for custom analysis."""
    if not scan_cache['data']:
        return pd.DataFrame()
    return pd.DataFrame(scan_cache['data'])

print("Quick Analysis Tools Ready")
print("=" * 50)
print("Usage:")
print("  show_by_symbol('NVDA')      - Show all opportunities for a symbol")
print("  show_top_n(10, 'ann_roc')   - Top 10 by ROC (or 'iv', 'rating')")
print("  group_by_symbol()           - Summary by ticker")
print("  export_cache('output.csv')  - Export to CSV")
print("  df = get_cache_df()         - Get DataFrame for custom analysis")

## Step 3: Quick Analysis Tools

Helper functions for deeper analysis. Run this cell to define the functions, then call them in any cell.